<a href="https://colab.research.google.com/github/Vitela-debug/Colab_S1_Clase-de-progralaci-n_273522_TVB/blob/main/Actividad_Calificada_Particion_Remuestreo_Arbol_273522.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad Calificada: Partición y Remuestreo con Árbol de Decisión

**Programación para Analítica Descriptiva y Predictiva — MIAAD, UACJ**


**Nombre:__Tania Vitela Barraza_**

**Matricula: ____273522__**

**Ficha técnica**

| | |
|---|---|
| **Modalidad** | Individual |
| **Entorno** | Google Colab |
| **Dataset** | Titanic (pasajeros del RMS Titanic, clasificación binaria) |
| **Modelo** | Árbol de Decisión (`DecisionTreeClassifier`) |
| **Puntaje total** | 100 puntos |
| **Fecha de entrega** | 10/24/2026 |

**Objetivo**

Aplicar partición hold-out estratificada y dos esquemas de validación cruzada (`StratifiedKFold` y `RepeatedStratifiedKFold`) para estimar el desempeño de un Árbol de Decisión, y justificar, con base en el sesgo y la varianza de la estimación, qué método reportarías como resultado final.

**Dataset**

El dataset contiene 891 pasajeros del Titanic, con la variable `survived` (0 = no sobrevivió, 1 = sobrevivió) como variable objetivo. Tiene valores ausentes genuinos en `age` (177 casos) y `embarked` (2 casos) — no son valores agregados artificialmente para esta actividad.

**Instrucciones generales**

- Completa únicamente las celdas marcadas con `# TODO`. No modifiques las celdas de preparación de datos ni los hiperparámetros ya indicados (`max_depth=4`, `random_state=0`), para que tus resultados sean comparables con los de tus compañeros.
- Cada actividad tiene una parte de código y una pregunta de interpretación en una celda de texto (Markdown). Responde directamente debajo de **Tu respuesta:**.
- No se califican fragmentos de código comentados o dejados sin completar (`# TODO` sin resolver).
- Antes de entregar, ejecuta todo el notebook de principio a fin (`Entorno de ejecución > Reiniciar y ejecutar todo` en Colab) y verifica que no queden errores.

**Entrega**

Sube a Moodle un archivo `.txt` con el enlace a tu notebook de Colab o GitHub (con permisos de lectura habilitados). La subida del archivo `.ipynb` es opcional, pero recomendada como respaldo.


## Preparación de los datos

Esta sección se proporciona completa: no es parte de la evaluación. Carga el dataset, selecciona las variables predictoras y aplica un tratamiento mínimo a los valores ausentes y a las variables categóricas, para que puedas concentrarte en la partición y el remuestreo.


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.precision", 3)

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

print(f"Observaciones: {df.shape[0]}, columnas: {df.shape[1]}")
print("Valores ausentes por columna (solo las que tienen alguno):")
print(df.isna().sum()[df.isna().sum() > 0])

# Variables predictoras: se dejan fuera las que son redundantes con 'survived'
# (alive, who, adult_male), las de muy alta cardinalidad (deck, con 688 ausentes)
# o duplicadas en otra forma (class es la versión en texto de pclass).
variables = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
X = df[variables].copy()
y = df["survived"].copy()

# Tratamiento mínimo, dado: imputar age con la mediana y embarked con la moda,
# y codificar las dos variables categóricas como enteros.
X["age"] = X["age"].fillna(X["age"].median())
X["embarked"] = X["embarked"].fillna(X["embarked"].mode()[0])
X["sex"] = X["sex"].map({"male": 0, "female": 1})
X["embarked"] = X["embarked"].map({"S": 0, "C": 1, "Q": 2})

print(f"\nVariables predictoras: {list(X.columns)}")
print(f"Distribución de la variable objetivo (survived): {dict(y.value_counts().sort_index())}")


Observaciones: 891, columnas: 15
Valores ausentes por columna (solo las que tienen alguno):
age            177
embarked         2
deck           688
embark_town      2
dtype: int64

Variables predictoras: ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
Distribución de la variable objetivo (survived): {0: np.int64(549), 1: np.int64(342)}


---
## Actividad 1 — Hold-out estratificado (20 pts)

**1.1 (15 pts).** Completa la partición hold-out: reserva el 20% de las observaciones para prueba, con estratificación por `y` y `random_state=42`. Ajusta un `DecisionTreeClassifier` con `max_depth=4` y `random_state=0` usando el conjunto de entrenamiento, y calcula su accuracy sobre el conjunto de prueba.


In [2]:
# TODO 1.1: completa la partición hold-out y el ajuste del modelo.
# Revisa las condiciones indicadas arriba (porcentaje de prueba, estratificación, random_state).
X_train, X_test, y_train, y_test = train_test_split(
    # TODO: agrega aquí los argumentos de train_test_split
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

arbol = DecisionTreeClassifier(max_depth=4, random_state=0)
# TODO: ajusta 'arbol' con los datos de entrenamiento

arbol.fit(X_train, y_train)

acc_holdout = accuracy_score(y_test, arbol.predict(X_test))  # TODO: calcula el accuracy de 'arbol' sobre el conjunto de prueba

# TODO: imprime el número de observaciones de entrenamiento, el número de observaciones
# de prueba y el accuracy obtenido en prueba (acc_holdout).
print("Observaciones de entrenamiento:", len(X_train))
print("Observaciones de prueba:", len(X_test))
print("Accuracy en prueba:", round(acc_holdout, 4))




Observaciones de entrenamiento: 712
Observaciones de prueba: 179
Accuracy en prueba: 0.7877


**1.2 (5 pts).** El Titanic tiene una proporción de supervivencia desbalanceada (aproximadamente 38% sobrevivió, 62% no). ¿Qué problema evita la estratificación (`stratify=y`) en este caso específico, y qué podría pasar con la partición si no se usara?

**Tu respuesta:**

La estratificación (stratify=y) evita que la proporción de sobrevivientes y no sobrevivientes cambie demasiado entre los datos de entrenamiento y prueba. En el Titanic, donde aproximadamente el 38 % sobrevivió y el 62 % no, sin estratificación podría ocurrir que uno de los conjuntos tuviera muchos más sobrevivientes o no sobrevivientes que el otro. Esto podría generar una evaluación menos representativa y hacer que el accuracy del modelo no refleje correctamente su desempeño.

---
---
* El código del ejercicio 1 divide los datos en 80 % para entrenamiento y 20 % para prueba mediante `test_size=0.20`. Usa `stratify=y` para mantener la proporción de las clases y `random_state=42 `para repetir la misma división. Después, crea un árbol de decisión con `max_depth=4` para limitar su profundidad y `random_state=0` para reproducir sus resultados. Finalmente, entrena el modelo y utiliza `accuracy_score(y_test, arbol.predict(X_test))` para calcular el porcentaje de predicciones correctas, mostrando también la cantidad de datos de entrenamiento y prueba.
---
---

---
## Actividad 2 — StratifiedKFold (25 pts)

**2.1 (15 pts).** Completa la validación cruzada con `StratifiedKFold`: 5 folds, `shuffle=True` y `random_state=42`. Usa `cross_validate` con un `DecisionTreeClassifier(max_depth=4, random_state=0)` nuevo (mismos hiperparámetros que en la Actividad 1) y reporta la media y la desviación estándar (`ddof=1`) del accuracy sobre los 5 folds.


In [3]:
# TODO 2.1: completa la validación cruzada con StratifiedKFold.

# Revisa las condiciones indicadas arriba (número de folds, shuffle, random_state).
cv_kfold = StratifiedKFold(
    # TODO: agrega aquí los argumentos de StratifiedKFold
    n_splits=5,
    shuffle=True,
    random_state=42
)


arbol_kfold = DecisionTreeClassifier(max_depth=4, random_state=0)
 # TODO: usa cross_validate con 'arbol_kfold', 'cv_kfold' y la métrica de accuracy
resultados_kfold = cross_validate(
    arbol_kfold,
    X,
    y,
    cv=cv_kfold,
    scoring="accuracy"
)


# TODO: extrae el arreglo de accuracy por fold a partir de 'resultados_kfold'
scores_kfold = resultados_kfold["test_score"]

# TODO: imprime el accuracy obtenido en cada fold, su media y su desviación estándar
# (usa ddof=1 para la desviación estándar, como se vio en la sesión).


media_kfold = scores_kfold.mean()
desv_kfold = scores_kfold.std(ddof=1)


print("Accuracy por fold:", np.round(scores_kfold, 4))
print("Media del accuracy:", round(media_kfold, 4))
print("Desviación estándar:", round(desv_kfold, 4))

Accuracy por fold: [0.832 0.826 0.792 0.809 0.837]
Media del accuracy: 0.8193
Desviación estándar: 0.0185


**2.2 (10 pts).** Compara la media del accuracy de `StratifiedKFold` con el accuracy único que obtuviste en la Actividad 1. ¿Por qué un solo valor de hold-out no es suficiente para saber si ese resultado es representativo del desempeño real del modelo?

**Tu respuesta:**

La media del accuracy de StratifiedKFold fue de 0.8193 (81.93 %), mientras que el accuracy del hold-out fue de 0.7877 (78.77 %). Esto significa que la validación cruzada obtuvo un resultado aproximadamente 3.16 puntos porcentuales mayor. Un solo resultado de hold-out no es suficiente porque depende de cómo se dividieron los datos y podría dar un resultado más alto o más bajo de lo habitual. En cambio, StratifiedKFold evalúa el modelo en cinco particiones diferentes y calcula la media, lo que permite tener una idea más representativa de su desempeño y observar cuánto varían los resultados.

---
---
* El código del ejercicio 2 realiza una validación cruzada para evaluar qué tan bien funciona el árbol de decisión. Con `n_splits=5`, divide los datos en cinco partes; `shuffle=True` mezcla los datos antes de dividirlos y `random_state=42 `permite repetir la misma división. Después, el modelo se entrena y se evalúa cinco veces, utilizando una parte diferente para probarlo en cada ocasión. Con `scoring="accuracy"`, se calcula el porcentaje de predicciones correctas en cada partición. Finalmente, se obtiene la media de los cinco resultados y, con `std(ddof=1)`, se calcula la desviación estándar muestral para conocer cuánto varía el accuracy entre las particiones.
---
---

---
## Actividad 3 — RepeatedStratifiedKFold (25 pts)

**3.1 (15 pts).** Completa la validación cruzada repetida: 5 folds, 5 repeticiones y `random_state=42`. Usa `cross_validate` con el mismo modelo (`DecisionTreeClassifier(max_depth=4, random_state=0)`) y reporta la media y la desviación estándar (`ddof=1`) sobre las 25 estimaciones resultantes.


In [4]:
# TODO 3.1: completa la validación cruzada repetida.
# Revisa las condiciones indicadas arriba (número de folds, número de repeticiones, random_state).
cv_repetido = RepeatedStratifiedKFold(
    # TODO: agrega aquí los argumentos de RepeatedStratifiedKFold
    n_splits=5,
    n_repeats=5,
    random_state=42
)

arbol_repetido = DecisionTreeClassifier(max_depth=4, random_state=0)
# TODO: usa cross_validate con 'arbol_repetido', 'cv_repetido' y la métrica de accuracy
resultados_repetido = cross_validate(
    arbol_repetido,
    X,
    y,
    cv=cv_repetido,
    scoring="accuracy"
 )


 # TODO: extrae el arreglo de accuracy a partir de 'resultados_repetido'

scores_repetido = resultados_repetido["test_score"]

# TODO: imprime el número de estimaciones obtenidas, su media y su desviación estándar (ddof=1).
media_repetido = scores_repetido.mean()
desv_repetido = scores_repetido.std(ddof=1)
print("Número de estimaciones:", len(scores_repetido))
print("Media del accuracy:", round(media_repetido, 4))
print("Desviación estándar:", round(desv_repetido, 4))

Número de estimaciones: 25
Media del accuracy: 0.8088
Desviación estándar: 0.0217


**3.2 (10 pts).** Compara la desviación estándar de `RepeatedStratifiedKFold` con la de `StratifiedKFold` (Actividad 2). ¿Qué fuente de variabilidad adicional captura la versión repetida que una sola corrida de `StratifiedKFold` no puede capturar?

**Tu respuesta:**

En StratifiedKFold, la desviación estándar fue de 0.0185, mientras que en RepeatedStratifiedKFold fue de 0.0217. La diferencia es que StratifiedKFold utiliza una sola configuración de cinco folds, mientras que RepeatedStratifiedKFold repite el proceso cinco veces, generando nuevas particiones de los datos. Esto hace que cambien las observaciones que se utilizan para entrenar y probar el modelo en cada repetición. Así, se puede observar cómo influye la composición de los datos en el accuracy y capturar una variabilidad que una sola ejecución de StratifiedKFold no muestra. En este caso, la desviación estándar fue ligeramente mayor en la validación repetida, lo que indica que los resultados variaron un poco más entre las 25 evaluaciones.

---
---
* El código del ejercicio 3 realiza una validación cruzada repetida para evaluar el desempeño del árbol de decisión. Con n_splits=5, divide los datos en cinco particiones, y con n_repeats=5, repite el proceso cinco veces, obteniendo un total de 25 evaluaciones. El parámetro random_state=42 permite reproducir las divisiones y scoring="accuracy" calcula el porcentaje de predicciones correctas en cada evaluación. Finalmente, se calcula la media de los 25 resultados para conocer el rendimiento promedio del modelo y, con std(ddof=1), se obtiene la desviación estándar muestral para observar cuánto varían los resultados entre las evaluaciones.
---
---

---
## Actividad 4 — Comparación y justificación (15 pts)

**4.1 (10 pts).** Completa la tabla comparativa con los tres resultados obtenidos. Para hold-out, al ser una sola estimación, no hay desviación estándar.


In [6]:
# TODO 4.1: completa la tabla comparativa con los tres métodos.
resumen = pd.DataFrame({
    "método": ["Hold-out", "StratifiedKFold", "RepeatedStratifiedKFold"],
    "n_estimaciones": [1, 5, 25],  # completa: número de estimaciones de cada método
    "media_accuracy": [0.7877, 0.8193, 0.8088],  # completa con los tres valores ya calculados
    "desv_est": [np.nan, 0.0185, 0.0217],  # completa; np.nan para hold-out
})
resumen


,método,n_estimaciones,media_accuracy,desv_est
0,Hold-out,1,0.788,NaN
1,StratifiedKFold,5,0.819,0.018
2,RepeatedStratifiedKFold,25,0.809,0.022


**4.2 (5 pts).** Si tuvieras que reportar un solo número como el desempeño esperado de este Árbol de Decisión en el dataset del Titanic, ¿cuál de los tres métodos usarías y por qué? Justifica tu respuesta en términos de sesgo y varianza de la estimación, no solo del valor de accuracy obtenido.

**Tu respuesta:**


Yo utilizaría RepeatedStratifiedKFold, porque realiza 25 evaluaciones con diferentes particiones de los datos y permite obtener una estimación más estable del desempeño esperado del modelo. Su media de accuracy fue de 80.88 %, con una desviación estándar de 0.0217. En términos de varianza, repetir las particiones ayuda a observar cuánto cambia el resultado según los datos utilizados para entrenar y probar el árbol. Hold-out tiene mayor incertidumbre al depender de una sola división, mientras que StratifiedKFold solo evalúa cinco particiones en una única corrida. Aunque RepeatedStratifiedKFold no garantiza por sí solo un menor sesgo ni elimina la dependencia entre las evaluaciones, ofrece una visión más completa de la variabilidad del modelo y una estimación menos dependiente de una sola partición.

---
---
* El código del ejercicio 4 crea una tabla comparativa de los tres métodos de evaluación: Hold-out, StratifiedKFold y RepeatedStratifiedKFold. Muestra cuántas estimaciones realiza cada método, la media del accuracy y la desviación estándar. En el caso de Hold-out, se coloca np.nan porque solo se obtiene una estimación y no se calcula la desviación estándar.
---
---

---
## Reflexión final (15 pts)

Responde en 6 a 10 líneas: ¿qué papel juega la elección del método de partición o remuestreo cuando se evalúa un modelo de clasificación, y qué factores del problema (tamaño del dataset, balance de clases, costo de entrenamiento) tomarías en cuenta para elegir uno sobre otro en un caso distinto al de esta actividad?

**Tu respuesta:**

La elección del método de partición es importante porque influye en qué tan confiable y representativo es el resultado del modelo. Hold-out es útil cuando se busca una evaluación rápida, mientras que la validación cruzada permite evaluar el modelo con diferentes particiones. RepeatedStratifiedKFold ayuda a observar mejor la variabilidad de los resultados al repetir el proceso varias veces. Si el dataset es pequeño, elegiría validación cruzada para aprovechar mejor los datos disponibles. Si las clases están desbalanceadas, usaría estratificación para mantener proporciones similares en cada partición.
También tomaría en cuenta el costo de entrenamiento, porque repetir muchas veces el proceso requiere más tiempo y recursos. Si el dataset es muy grande o el modelo tarda mucho en entrenar, podría preferir hold-out o una validación cruzada con menos folds. En conclusión, elegiría el método según el tamaño de los datos, el balance de clases y los recursos disponibles, buscando una evaluación confiable sin gastar tiempo innecesariamente.

---
## Rúbrica

| Criterio | Excelente | Bueno | Insuficiente | No entregado / Incorrecto | Puntos |
|---|---|---|---|---|---|
| **1.1 Hold-out estratificado** | Partición y ajuste correctos; accuracy calculado e interpretado correctamente. | Partición y ajuste correctos, con un error menor (p. ej. un argumento incompleto) que no afecta el resultado final. | Partición o ajuste con errores que alteran el resultado (p. ej. sin estratificar, sin `random_state`). | Celda sin completar o con errores que impiden su ejecución. | 15 |
| **1.2 Interpretación de la estratificación** | Explica correctamente el riesgo de desbalance en la partición y por qué `stratify` lo evita. | Explica la idea general pero de forma incompleta o imprecisa. | Respuesta genérica, sin relación con el desbalance de clases del dataset. | Sin respuesta. | 5 |
| **2.1 StratifiedKFold** | Configuración correcta de `cv` y `cross_validate`; media y desviación estándar reportadas con `ddof=1`. | Correcto, con un error menor que no cambia el resultado (p. ej. `ddof` por defecto). | Configuración incompleta o métrica mal calculada. | Celda sin completar o con errores que impiden su ejecución. | 15 |
| **2.2 Comparación hold-out vs. k-fold** | Explica con claridad por qué una sola partición no es representativa y cómo k-fold promedia sobre varias particiones. | Identifica la idea central pero sin justificarla completamente. | Comparación superficial, sin referirse a la variabilidad entre particiones. | Sin respuesta. | 10 |
| **3.1 RepeatedStratifiedKFold** | Configuración correcta de `cv` y `cross_validate`; media y desviación estándar reportadas sobre las 25 estimaciones. | Correcto, con un error menor que no cambia el resultado. | Configuración incompleta (p. ej. `n_repeats` incorrecto) o métrica mal calculada. | Celda sin completar o con errores que impiden su ejecución. | 15 |
| **3.2 Fuente de variabilidad adicional** | Identifica correctamente que la repetición varía la configuración de folds entre repeticiones, no solo las observaciones dentro de un fold. | Identifica que hay más variabilidad capturada, sin precisar el mecanismo. | Respuesta genérica o incorrecta sobre qué varía entre repeticiones. | Sin respuesta. | 10 |
| **4.1 Tabla comparativa** | Tabla completa y correcta, con `np.nan` o vacío en la desviación estándar de hold-out. | Tabla completa con un valor incorrecto o mal ubicado. | Tabla incompleta o con varios valores incorrectos. | Celda sin completar. | 10 |
| **4.2 Justificación del método elegido** | Justifica la elección con el argumento de sesgo/varianza de la estimación, no solo con el valor numérico de accuracy. | Justifica la elección, pero basándose solo en qué método dio el accuracy más alto. | Elige un método sin justificación relacionada con sesgo o varianza. | Sin respuesta. | 5 |
| **Reflexión final** | Identifica con claridad los factores relevantes (tamaño del dataset, balance de clases, costo) y los conecta con el método de partición o remuestreo adecuado. | Identifica algunos factores relevantes, de forma parcial. | Reflexión genérica, sin relación específica con los factores de decisión vistos en la sesión. | Sin respuesta. | 15 |

**Nota.** La penalización por entrega tardía es del 5% por día calendario, hasta 7 días, después de lo cual la actividad recibe 0 puntos. Esta penalización se aplica sobre el puntaje ya obtenido con la rúbrica, no como parte de ella.
